# TF-IDF classifiers
Train Logistic Regression and Linear SVM on a labeled CSV, evaluate on an untouched test split, save artifacts, and inspect errors. Use separate files/splits prepared in notebook 01.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
import joblib
TRAIN = pd.read_csv('../data/train/dataset.csv').dropna(subset=['text','sentiment'])
TEST = pd.read_csv('../data/test/dataset.csv').dropna(subset=['text','sentiment'])
def build(classifier):
    return Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=50000)), ('classifier', classifier)])
models = {'tfidf_lr': build(LogisticRegression(max_iter=1000, class_weight='balanced')), 'tfidf_svm': build(LinearSVC(class_weight='balanced'))}

In [ ]:
for name, model in models.items():
    model.fit(TRAIN['text'].astype(str), TRAIN['sentiment'].astype(str))
    predicted = model.predict(TEST['text'].astype(str))
    print('\n', name)
    print(classification_report(TEST['sentiment'].astype(str), predicted, zero_division=0))
    print(confusion_matrix(TEST['sentiment'].astype(str), predicted))
    output = Path('../models/sentiment') / f'{name}.joblib'
    output.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({'pipeline': model, 'task': 'sentiment', 'family': name}, output)
    errors = TEST.loc[TEST['sentiment'].astype(str).to_numpy() != predicted, ['text','sentiment']].copy()
    errors['predicted'] = predicted[TEST['sentiment'].astype(str).to_numpy() != predicted]
    display(errors.head(30))